## SECURITY PROVISIONING: Authorizing the Service Principal (SPN)
-- Purpose: To enable automated Job execution without manual user intervention.

In [0]:
%sql

-- 1. CATALOG LEVEL: Establishing the root of trust.
-- USAGE allows the SPN to see the catalog; ALL PRIVILEGES allows it to manage lifecycle events.
GRANT USAGE ON CATALOG ecommerce_analytics_dev TO `b3b580fc-1cd0-4cb3-a176-4bcf4d49945e`;
GRANT ALL PRIVILEGES ON CATALOG ecommerce_analytics_dev TO `b3b580fc-1cd0-4cb3-a176-4bcf4d49945e`;

-- 2. SCHEMA LEVEL: Defining the workspace for our Medallion layers.
-- Ensures the SPN can physically write the Delta tables for Bronze, Silver, and Gold.
GRANT ALL PRIVILEGES ON SCHEMA ecommerce_analytics_dev.bronze_layer TO `b3b580fc-1cd0-4cb3-a176-4bcf4d49945e`;

-- 3. VOLUME LEVEL: Authorizing Checkpoint Persistence.
-- Streaming state (checkpoints) must be saved to a Volume. 
-- Without READ/WRITE access here, the pipeline cannot track which of the 110M rows were processed.
GRANT ALL PRIVILEGES ON VOLUME ecommerce_analytics_dev.bronze_layer.checkpoints TO `b3b580fc-1cd0-4cb3-a176-4bcf4d49945e`;

-- 4. VERIFICATION: Audit trail to confirm permissions were applied correctly.
SHOW GRANTS ON CATALOG ecommerce_analytics_dev;

## Pipeline Permission Fix (Python)
This uses the Databricks API to tell the workspace: "Let this Service Principal manage my existing pipeline.

In [0]:
import requests

# 1. Setup Variables
# These are pulled directly from your workspace context
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Your specific ID from the URL you provided
pipeline_id = "72a90a53-8e66-4a11-be7a-10707242236f" 
spn_id = "b3b580fc-1cd0-4cb3-a176-4bcf4d49945e"

# 2. Define the Permission Payload
payload = {
    "access_control_list": [
        {
            "service_principal_name": spn_id,
            "permission_level": "CAN_MANAGE"
        }
    ]
}

# 3. Send the API Request
print(f"Updating permissions for Pipeline: {pipeline_id}...")
response = requests.patch(
    f"{workspace_url}/api/2.0/permissions/pipelines/{pipeline_id}",
    headers={"Authorization": f"Bearer {token}"},
    json=payload
)

# 4. Check Result
if response.status_code == 200:
    print("SUCCESS: The Service Principal now has CAN_MANAGE permissions.")
else:
    print(f"FAILED: Status Code {response.status_code}")
    print(response.text)